# Acemi Kaggle runner — Adım 2: UCI durdurma doğrulaması

Sabit referansla UCI düzeltmesini karşılaştırır: sabit `db058b3` commit'i ve güncel checkout için test paketi, test özeti/günlüğü ve aynı derinlikte 10 konumluk benchmark. Commit, motor ağacı, Python sürümü, bağımlılıklar ve donanım bilgisi sonuçlarla birlikte saklanır. Sonuçlar `data/baselines/<oturum>/reference` ve `current` altında yeni klasörlere yazılır; önceki ölçümler değiştirilmez.

Hücreleri sırayla çalıştırın; Internet ve Kaggle `GITHUB_TOKEN` secret'ı açık olmalı. Yayınlama yardımcısı testlerden önce tanımlanır: test başarısızlığında da günlükler GitHub'a gönderilmeye çalışılır. Push başarısızsa dosyalar Kaggle'da kalır ve son hücreyle tekrar gönderilebilir.

Bu adım UCI giriş/worker yaşam döngüsünü değiştirir; arama algoritmasını ve değerlendirmeyi değiştirmez. Bu nedenle konum kümesi, FEN, skor, tamamlanan derinlik, düğüm ve en iyi hamle eşitliği beklenir. Sonraki davranış düzeltmelerinde değişen sonuçlar ayrıca değerlendirilmelidir. Benchmark süresi tek koşudur; Elo veya kesin hız kazancı anlamına gelmez.

`RUN_VALIDATION_MATCHES=False` varsayılanıyla yalnızca referans toplanır. İsteğe bağlı self-play ve Stockfish kontrol maçları aynı notebook'ta korunmuştur; çalıştırmak için bu ayarı `True` yapın. Eski artımlı eval A/B kazanma şartı bu referans akışından kaldırılmıştır.

İsteğe bağlı kısa kontroller: `SELF_PLAY_GAMES=10`, her Stockfish seviyesi için `STOCKFISH_GAMES=20`; toplam 50 oyun. Bunlar hata yakalamak içindir; Elo artışını kanıtlamaz. Varsayılan ayarda maç sayısı sıfırdır.

Çalıştırma dalı varsayılan olarak `codex/kaggle-baseline` seçilidir. Kurulum hücresi mevcut Kaggle checkout’unu bu dala geçirir ve testlere başlamadan `tools/baseline.py` dosyasını kontrol eder. Notebook’u GitHub’dan içe aktarırken seçilen dal ile bu ayar aynı olmalıdır.

`VALIDATION_LABEL="step2-uci-stop"` yeni kayıtları etiketler. Güncel paket `tests/test_uci_async.py` içindeki worker ve gerçek UCI pipe regresyonlarını da çalıştırır. Uzun test/benchmark bu Kaggle oturumunda yürütülür; yerel bilgisayarda maç başlatılması gerekmez. Önceki referans kayıtları korunur.


In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "codex/kaggle-baseline"  # Geliştirme dalı; main ile en son birleştirilecek.
WORKSPACE = "/kaggle/working/acemi"
BASELINE_COMMIT = "db058b344df8759810e957f11a58b4149055fe49"
BASELINE_DEPTH = 4
RUN_VALIDATION_MATCHES = False
VALIDATION_LABEL = "step2-uci-stop"

# Short smoke checks only; these counts do not establish playing strength.
SELF_PLAY_GAMES = 10
STOCKFISH_GAMES = 20  # Per Stockfish level; colour-swapped pairs.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

def run_quietly(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        details = "\n".join(part for part in (result.stdout, result.stderr) if part)
        raise RuntimeError(f"Command failed: {' '.join(args)}\n{details[-4000:]}")
    return result.stdout

repo_url = GITHUB_REPO_URL.strip()
if not repo_url.startswith("https://github.com/") or "YOUR_ACCOUNT" in repo_url:
    raise ValueError("Set GITHUB_REPO_URL to your repository's HTTPS clone URL.")
if "@" in repo_url.split("github.com/", 1)[-1].split("/", 1)[0]:
    raise ValueError("Do not put credentials in GITHUB_REPO_URL. Use the GITHUB_TOKEN Kaggle Secret.")
workspace = Path(WORKSPACE)
if not workspace.exists():
    run_quietly(["git", "clone", "--branch", GITHUB_BRANCH, repo_url, str(workspace)])
else:
    # Pulling another branch does not switch an existing checkout to that branch.
    # Keep local records; git switch/merge will stop rather than overwrite conflicts.
    run_quietly(["git", "-C", str(workspace), "fetch", "origin", GITHUB_BRANCH])
    local_branch = subprocess.run(
        ["git", "-C", str(workspace), "show-ref", "--verify", "--quiet", f"refs/heads/{GITHUB_BRANCH}"],
        text=True, capture_output=True,
    )
    if local_branch.returncode == 0:
        run_quietly(["git", "-C", str(workspace), "switch", GITHUB_BRANCH])
    else:
        run_quietly(["git", "-C", str(workspace), "switch", "--create", GITHUB_BRANCH,
                     "--track", f"origin/{GITHUB_BRANCH}"])
    run_quietly(["git", "-C", str(workspace), "merge", "--ff-only", f"origin/{GITHUB_BRANCH}"])
os.chdir(workspace)
actual_branch = run_quietly(["git", "-C", str(workspace), "branch", "--show-current"]).strip()
if actual_branch != GITHUB_BRANCH:
    raise RuntimeError(f"Wrong checkout branch: {actual_branch}; expected {GITHUB_BRANCH}")
if not (workspace / "tools" / "baseline.py").is_file():
    raise RuntimeError(
        f"tools/baseline.py is missing on {actual_branch}. Import the latest notebook from "
        "codex/kaggle-baseline and rerun the settings and repository setup cells."
    )
print(f"Repository ready on branch {actual_branch}; tools/baseline.py found.")

print(run_quietly(["git", "-C", str(workspace), "log", "-1", "--format=%h %s"]).strip())
version = subprocess.run(
    ["python", "-c", "import engine; print(engine.__version__)"], text=True, capture_output=True
)
print(f"Engine version: {version.stdout.strip()}")
if version.returncode:
    raise RuntimeError(version.stderr)

# Latest commit that touched code/data files that define the engine, not the data records.
CODE_COMMIT = run_quietly([
    "git", "-C", str(workspace), "log", "-1", "--format=%H", "--",
    "engine", "tools", "tests", "openings", "notebooks", ".github",
]).strip() or run_quietly(["git", "-C", str(workspace), "rev-parse", "HEAD"]).strip()
print(f"Code commit pinned for every run: {CODE_COMMIT[:7]}")

In [ ]:
# Tests require tool dependencies. Stockfish is only needed for optional matches.
run_quietly(["python", "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"])
stockfish_bin = None
if RUN_VALIDATION_MATCHES:
    run_quietly(["apt-get", "update", "-qq"])
    run_quietly(["apt-get", "install", "-y", "-qq", "stockfish"])
    stockfish_bin = shutil.which("stockfish") or ("/usr/games/stockfish" if Path("/usr/games/stockfish").exists() else None)
    if stockfish_bin is None:
        raise RuntimeError("Stockfish package installed but executable was not found.")
print("Tools installed; optional validation matches:", RUN_VALIDATION_MATCHES)


In [ ]:
# Push helper is defined before tests so failed reference records can be published too.
# and in the last cell. Anything committed earlier but not yet pushed is pushed too.
import base64
from kaggle_secrets import UserSecretsClient

def publish_results(message):
    try:
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is unavailable; add it under Add-ons / Secrets.") from None
    if not token:
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is empty.")
    auth_value = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": auth_value,
        "GIT_TERMINAL_PROMPT": "0",
    })

    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=git_env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stderr + result.stdout).replace(token, "[REDACTED]")
            raise RuntimeError(f"Git command failed ({args[0]}): {details[-1500:]}")
        return result

    git("config", "user.name", "Acemi Kaggle Runner")
    git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
    tracked = [name for name in ("data/runs", "data/index.csv", "data/positions", "data/bench", "data/baselines") if (workspace / name).exists()]
    git("add", "--", *tracked)
    if git("diff", "--cached", "--quiet", check=False).returncode != 0:
        git("commit", "-m", message)
        print(f"Committed: {message}")
    git("fetch", "origin", GITHUB_BRANCH)
    unpushed = int(git("rev-list", "--count", "FETCH_HEAD..HEAD").stdout.strip())
    if unpushed == 0:
        print("Nothing to push: GitHub already has every local record.")
        return
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()) > 0:
        git("rebase", "FETCH_HEAD")  # GitHub moved on since this session started.
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    local_head = git("rev-parse", "HEAD").stdout.strip()
    remote_line = git("ls-remote", "origin", f"refs/heads/{GITHUB_BRANCH}").stdout.split()
    if not remote_line or remote_line[0] != local_head:
        raise RuntimeError("Push finished but GitHub does not show the new commit; run this cell again.")
    print(f"Pushed to GitHub ({unpushed} commit(s)); {GITHUB_BRANCH} is now at {local_head[:7]}.")

In [ ]:
# Freeze the reference source. Never use a moving branch name as the reference.
baseline_workspace = Path("/kaggle/working/acemi-reference")
if not baseline_workspace.exists():
    run_quietly(["git", "clone", "--depth", "1", "--no-checkout", GITHUB_REPO_URL.strip(), str(baseline_workspace)])
run_quietly(["git", "-C", str(baseline_workspace), "fetch", "--depth", "1", "origin", BASELINE_COMMIT])
run_quietly(["git", "-C", str(baseline_workspace), "checkout", "--detach", BASELINE_COMMIT])
baseline_head = run_quietly(["git", "-C", str(baseline_workspace), "rev-parse", "HEAD"]).strip()
if baseline_head != BASELINE_COMMIT:
    raise RuntimeError(f"Baseline mismatch: {baseline_head} != {BASELINE_COMMIT}")
print(f"Reference source pinned: {baseline_head}")


In [ ]:
# Record both builds on this Kaggle machine. Every execution creates a new session.
import json
import uuid
from datetime import datetime, timezone

if not (workspace / "tools" / "baseline.py").is_file():
    raise RuntimeError("tools/baseline.py is missing; rerun the repository setup cell before collection.")

session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
session_root = workspace / "data" / "baselines" / session_id
reference_dir = session_root / "reference"
current_dir = session_root / "current"

for label, source_root, output_dir in (
    (f"{VALIDATION_LABEL}-reference", baseline_workspace, reference_dir),
    (f"{VALIDATION_LABEL}-current", workspace, current_dir),
):
    if source_root == workspace:
        latest_code = run_quietly([
            "git", "-C", str(workspace), "log", "-1", "--format=%H", "--",
            "engine", "tools", "tests", "openings", "notebooks", ".github",
        ]).strip()
        if latest_code != CODE_COMMIT:
            raise RuntimeError("Code changed while publishing the reference; restart from setup to pin the new code.")
    command = [
        "python", "-m", "tools.baseline", "--source-root", str(source_root),
        "--out-dir", str(output_dir), "--depth", str(BASELINE_DEPTH), "--label", label,
    ]
    if source_root == baseline_workspace:
        command += ["--expected-commit", BASELINE_COMMIT]
    print("$", " ".join(command), flush=True)
    result = subprocess.run(command, cwd=workspace, text=True, capture_output=True)
    print(result.stdout, end="")
    print(result.stderr, end="")
    try:
        publish_results(f"data: record {label} ({session_id})")
    except RuntimeError as error:
        print("PUSH FAILED. Records remain in Kaggle; retry the final cell.")
        print(error)
    if result.returncode:
        raise RuntimeError(f"Reference collection failed for {label}; inspect {output_dir}. Logs were retained.")

comparison = subprocess.run(
    ["python", "-m", "tools.bench", "--compare", str(reference_dir / "bench.json"), str(current_dir / "bench.json")],
    cwd=workspace, text=True, capture_output=True,
)
comparison_text = comparison.stdout + comparison.stderr
(session_root / "comparison.log").write_text(comparison_text, encoding="utf-8")
(session_root / "session.json").write_text(json.dumps({
    "schema_version": 1,
    "validation_label": VALIDATION_LABEL,
    "reference_commit": BASELINE_COMMIT,
    "current_code_commit": CODE_COMMIT,
    "depth": BASELINE_DEPTH,
    "search_identical": comparison.returncode == 0,
    "reference": "reference",
    "current": "current",
}, indent=2) + "\n", encoding="utf-8")
print(comparison_text, end="")
try:
    publish_results(f"data: record reference comparison ({session_id})")
except RuntimeError as error:
    print("PUSH FAILED. Records remain in Kaggle; retry the final cell.")
    print(error)
if comparison.returncode:
    raise RuntimeError("Search differs from the pinned reference; inspect comparison.log before changing the engine.")
print(f"{VALIDATION_LABEL} complete: {session_root}")


In [ ]:
if RUN_VALIDATION_MATCHES:
    # Short self-play smoke at 100 ms, paired openings, book off.
    subprocess.run(
        [
            "python", "-m", "tools.match",
            "--games", str(SELF_PLAY_GAMES),
            "--movetime-ms", "100",
            "--openings", "openings/acemi-64.txt",
            "--opponent-cmd", "python -u -m engine",
            "--opponent-cwd", str(workspace),
            "--git-commit", CODE_COMMIT,
        ],
        check=True,
    )
    try:
        publish_results(f"data: self-play smoke ({SELF_PLAY_GAMES} games at 100 ms)")
    except RuntimeError as error:
        print("PUSH FAILED. The records are still in the Kaggle session; run the last cell to retry.")
        print(error)
else:
    print("Optional match skipped; RUN_VALIDATION_MATCHES=False.")


In [ ]:
if RUN_VALIDATION_MATCHES:
    # Short smoke 1/2 vs Stockfish 1500 at 500 ms, book off.
    subprocess.run(
        [
            "python", "-m", "tools.match",
            "--games", str(STOCKFISH_GAMES),
            "--movetime-ms", "500",
            "--uci-elo", "1500",
            "--openings", "openings/acemi-64.txt",
            "--stockfish-path", stockfish_bin,
            "--git-commit", CODE_COMMIT,
        ],
        check=True,
    )
    try:
        publish_results(f"data: smoke vs SF1500 at 500 ms ({STOCKFISH_GAMES} games)")
    except RuntimeError as error:
        print("PUSH FAILED. The records are still in the Kaggle session; run the last cell to retry.")
        print(error)
else:
    print("Optional match skipped; RUN_VALIDATION_MATCHES=False.")


In [ ]:
if RUN_VALIDATION_MATCHES:
    # Short smoke 2/2 vs Stockfish 1320 at 250 ms, book off.
    subprocess.run(
        [
            "python", "-m", "tools.match",
            "--games", str(STOCKFISH_GAMES),
            "--movetime-ms", "250",
            "--uci-elo", "1320",
            "--openings", "openings/acemi-64.txt",
            "--stockfish-path", stockfish_bin,
            "--git-commit", CODE_COMMIT,
        ],
        check=True,
    )
    try:
        publish_results(f"data: smoke vs SF1320 at 250 ms ({STOCKFISH_GAMES} games)")
    except RuntimeError as error:
        print("PUSH FAILED. The records are still in the Kaggle session; run the last cell to retry.")
        print(error)
else:
    print("Optional match skipped; RUN_VALIDATION_MATCHES=False.")


In [ ]:
# Retry publishing any reference or optional match records that remain local.
publish_results("data: record baseline and validation results")
